# NaijaCode RAG Assistant 🇳🇬

Week 5 Exercise — Building a RAG (Retrieval Augmented Generation) assistant

This is a knowledge worker chatbot for **NaijaCode** — a fictional Nigerian tech company.
It uses RAG to answer questions about the company's employees, products, and policies.

What we're doing:
1. Loading documents from our knowledge base
2. Chunking them up
3. Creating vector embeddings with HuggingFace
4. Storing in Chroma vector database
5. Building a RAG chat with Gradio UI

By **Victor Conqueror** 🚀

In [ ]:
# Data Setup — Download knowledge base if missing
import os
import zipfile
import urllib.request

KB_DIR = "knowledge-base"
ZIP_URL = "https://drive.google.com/uc?export=download&id=1VoCSN31onJCiYfDlxOH2PL5YQyngN7Sh"
ZIP_FILE = "knowledge-base.zip"

if not os.path.exists(KB_DIR):
    print(f"Knowledge base not found. Downloading...")
    try:
        urllib.request.urlretrieve(ZIP_URL, ZIP_FILE)
        # Create the directory if it doesn't exist
        os.makedirs(KB_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extractall(KB_DIR)
        os.remove(ZIP_FILE)
        print("Knowledge base downloaded and extracted!")
    except Exception as e:
        print(f"Error downloading data: {e}")
        print("Please ensure you have an internet connection and the link is valid.")
else:
    print("Knowledge base already exists.")

In [ ]:
import os
import glob
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
# Setup

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — oya go add am to your .env!")

MODEL = "gpt-4o-mini"
DB_NAME = "naijacode_vector_db"

## Step 1: Load the Knowledge Base

We have markdown files about:
- **employees/** — Adaeze (CTO), Chinedu (Backend Lead), Fatima (Head of Product), Seun (CEO), Amina (Frontend Lead)
- **products/** — CodeNaija IDE, NaijaCode Marketplace, NaijaCode AI
- **company/** — Company overview, Policies & benefits

In [ ]:
# Load all documents from the knowledge base

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents from the knowledge base")
for doc in documents:
    print(f"  - {doc.metadata.get('source', 'unknown')} ({doc.metadata['doc_type']})")

In [ ]:
# Let's peek at one document

print(documents[0].page_content[:500])

## Step 2: Chunk the Documents

We split the documents into smaller chunks so the vector search can find the most relevant bits.
Using `RecursiveCharacterTextSplitter` from LangChain — same as the course.

In [ ]:
# Split into chunks

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print(f"\nFirst chunk preview:\n{chunks[0].page_content[:300]}...")

## Step 3: Create Embeddings & Store in Chroma

Using HuggingFace's `all-MiniLM-L6-v2` model — it's free and works great.
Storing everything in a Chroma vector database.

In [ ]:
# Create embeddings and store in Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Delete old database if it exists
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

print(f"Vector store created with {vectorstore._collection.count()} documents!")

## Step 4: Build the RAG Pipeline

Now we connect everything:
1. User asks a question
2. We search the vector store for relevant chunks
3. We pass the chunks + question to the LLM
4. LLM gives a grounded answer

In [ ]:
# Set up retriever and LLM

retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
SYSTEM_PROMPT = """
You are the NaijaCode Assistant — a helpful, friendly chatbot for employees of NaijaCode, a Nigerian tech company.
You answer questions about NaijaCode's employees, products, and company policies.

Rules:
1. Use the provided context to answer questions accurately.
2. Be friendly and professional. You can throw in a little Nigerian flavor if appropriate.
3. If you don't know the answer or it's not in the context, say so honestly.
4. Give concise but complete answers.

Context from the NaijaCode knowledge base:
{context}
"""

In [ ]:
def answer_question(question, history):
    """RAG pipeline: retrieve context, then generate answer"""
    
    # Retrieve relevant chunks
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    
    # Build the prompt with context
    system_prompt = SYSTEM_PROMPT.format(context=context)
    
    # Call the LLM
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ])
    
    return response.content

## Let's test it!

In [ ]:
# Test some questions

test_questions = [
    "Who is the CTO of NaijaCode?",
    "What is the CodeNaija IDE?",
    "How many days of annual leave do employees get?",
    "Who won the TechHer Nigeria award?",
    "What is the pricing for NaijaCode Marketplace templates?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q, [])}")
    print("-" * 80)

## Step 5: Gradio Chat Interface

Time to make it look sharp with a proper chat UI!

In [ ]:
gr.ChatInterface(
    fn=answer_question,
    title="🇳🇬 NaijaCode Assistant",
    description="Ask me anything about NaijaCode — our employees, products, and policies!",
    examples=[
        "Who founded NaijaCode?",
        "What products does NaijaCode offer?",
        "Tell me about the engineering team",
        "What is the leave policy?",
        "How much funding has NaijaCode raised?",
    ],
    type="messages",
).launch(inbrowser=True)

## That's it! 🎉

We built a full RAG pipeline from scratch:
1. ✅ Loaded documents from our custom knowledge base
2. ✅ Chunked them for efficient retrieval
3. ✅ Created vector embeddings with HuggingFace
4. ✅ Stored them in Chroma
5. ✅ Built a RAG chat with Gradio

### Ideas to extend this:
- Add more knowledge base files (contracts, FAQs, etc.)
- Try different embedding models (OpenAI, Cohere)
- Add reranking like we did on Day 5
- Build evaluation tests to measure accuracy
- Add a Pidgin English mode for the assistant 😄